In [0]:
%run /Workspace/Users/ashishbudz@gmail.com/databricks_pipeline/1_setup/utilities


In [0]:
print(bronze_schema)
print(silver_schema)
print(gold_schema)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
#Creating widgets
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

#Retrieving values of widgets
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
print(catalog)
print(data_source)

In [0]:
#Path from where to read the customers data from
base_path = f's3://sportsbar-dp-child-company-prac/{data_source}/*.csv'


In [0]:
#Reading the data from the S3 bucket using the base path
df =( 
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:

display(df)

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')
    